<a href="https://colab.research.google.com/github/nikamsudarshan/dog_vs_cat_classifier/blob/main/Dog_vs_Cat_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import libraries
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np

# 1. Download and load the dataset (This takes a minute to download ~800MB)
# We split it into 80% for training and 20% for validation
(raw_train, raw_validation), metadata = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:]'],
    with_info=True,
    as_supervised=True,
)

# 2. Preprocessing Function
# MobileNetV2 expects images to be exactly 224x224 pixels and normalized between -1 and 1.
IMG_SIZE = 224

def format_example(image, label):
    image = tf.cast(image, tf.float32)
    image = (image / 127.5) - 1.0 # Normalize to [-1, 1]
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    return image, label

# Apply the formatting and batch the data for faster training
BATCH_SIZE = 32
train_batches = raw_train.map(format_example).shuffle(1000).batch(BATCH_SIZE)
validation_batches = raw_validation.map(format_example).batch(BATCH_SIZE)

print("Data loaded and preprocessed successfully!")

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/cats_vs_dogs/incomplete.J9B4MJ_4.0.1/cats_vs_dogs-train.tfrecord-[0-9][0-9…

Dataset cats_vs_dogs downloaded and prepared to /root/tensorflow_datasets/cats_vs_dogs/4.0.1. Subsequent calls will reuse this data.
Data loaded and preprocessed successfully!


In [ ]:
IMG_SHAPE = (IMG_SIZE, IMG_SIZE, 3) # 224x224 with 3 color channels (RGB)

# 1. Load the pre-trained base model (MobileNetV2)
base_model = tf.keras.applications.MobileNetV2(input_shape=IMG_SHAPE,
                                               include_top=False, # We don't want the original 1000-class output
                                               weights='imagenet')

# 2. Freeze the base model so its weights don't change during our training
base_model.trainable = False

# 3. Build our custom classification head on top
model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(), # Flattens the output
    tf.keras.layers.Dropout(0.2),             # Prevents overfitting
    tf.keras.layers.Dense(1, activation='sigmoid') # Sigmoid gives a probability between 0 and 1
])

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
# Compile the model
# We use Binary Crossentropy because there are only two choices (0 or 1)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Train the model
history = model.fit(train_batches,
                    epochs=3,
                    validation_data=validation_batches)

Epoch 1/3
582/582 ━━━━━━━━━━━━━━━━━━━━ 1040s 2s/step - accuracy: 0.9199 - loss: 0.2481 - val_accuracy: 0.9714 - val_loss: 0.1072
Epoch 2/3
582/582 ━━━━━━━━━━━━━━━━━━━━ 1019s 2s/step - accuracy: 0.9778 - loss: 0.0898 - val_accuracy: 0.9819 - val_loss: 0.0676
Epoch 3/3
582/582 ━━━━━━━━━━━━━━━━━━━━ 987s 2s/step - accuracy: 0.9823 - loss: 0.0638 - val_accuracy: 0.9843 - val_loss: 0.0537


In [ ]:
!pip install gradio -q
import gradio as gr
import cv2

def classify_pet(image):
    """Preprocesses user image and predicts Dog or Cat"""
    # 1. Resize to the 224x224 expected by MobileNetV2
    img = cv2.resize(image, (IMG_SIZE, IMG_SIZE))

    # 2. Normalize colors to [-1, 1]
    img = (img / 127.5) - 1.0

    # 3. Reshape for the model (Add a batch dimension: (1, 224, 224, 3))
    img = np.expand_dims(img, axis=0)

    # 4. Predict
    prediction = model.predict(img)
    probability = prediction[0][0] # Float between 0.0 and 1.0

    # 5. Format output
    # Since Dog is 1 and Cat is 0
    if probability > 0.5:
        confidence = probability * 100
        return f"🐶 It's a Dog! (Confidence: {confidence:.2f}%)"
    else:
        confidence = (1.0 - probability) * 100
        return f"🐱 It's a Cat! (Confidence: {confidence:.2f}%)"

# Build the UI
interface = gr.Interface(
    fn=classify_pet,
    inputs=gr.Image(type="numpy"), # Image upload box
    outputs="text",
    title="Dog vs. Cat Image Classifier",
    description="Upload a photo of a dog or cat, and the MobileNetV2 Deep Learning model will classify it!"
)

# Launch the app
interface.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a97fa8a1acc747c060.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/PIL/ImageFile.py", line 370, in load
    s = read(read_bytes)
        ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/PIL/PngImagePlugin.py", line 994, in load_read
    cid, pos, length = self.png.read()
                       ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/PIL/PngImagePlugin.py", line 176, in read
    length = i32(s)
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/PIL/_binary.py", line 95, in i32be
    return unpack_from(">I", c, o)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^
struct.error: unpack_from requires a buffer of at least 4 bytes for unpacking 4 bytes at offset 0 (actual buffer size is 0)

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_proces

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 170ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
